# အခန်း ၁၃ - Cognee အသိအမှတ်ပြု အကြောင်းအရာဇယားနှင့် အေးဂျင့်မှတ်ဉာဏ်


## သတ်မှတ်ခြင်း

ဒီ notebook က [**Cognee**](https://www.cognee.ai/) သိပ္ပံဖြစ်ရပ်များနှင့် **Microsoft Agent Framework** (MAF) ကို အသုံးပြု၍ တည်ငြိမ်သော စွမ်းမွှားသည့် **coding assistant** တစ်ခုကို ဘယ်လိုတည်ဆောက်ရမယ်ဆိုတာကို ဖော်ပြပေးမှာဖြစ်ပါတယ်။

Cognee ဟာ ဖွဲ့စည်းပုံမရှိတဲ့ စာသားတွေကို အဖွဲ့တစ်ခုချင်းစီစစ်ဆေးနိုင်တဲ့ သိပ္ပံဖြစ်ရပ်များတစ်ခုဖြစ်အောင် ပြောင်းလဲပေးပြီး vector embeddings ကို ထောက်ပံ့သည့်အတွက် သင့်အေဂျင့်ကို ဆက်စပ်မှုကို အသိအမှတ်ပြုသည့် ရှည်လျားသော အမှတ်တရတစ်ခု ပေးစွမ်းနိုင်ပါတယ်။

### သင်တတ်နိုင်မယ့်အရာတွေ
1. **သိပ္ပံဖြစ်ရပ်များတည်ဆောက်ခြင်း**: တီထွင်သူများ၏ ပရိုဖိုင်များနှင့် အကောင်းဆုံးအတွေ့အကြုံများကို ဖွဲ့စည်းပြုပြင်နိုင်ပြီး စစ်ဆေးနိုင်သော သိပ္ပံဖြစ်ရပ်များအဖြစ် ပြောင်းလဲခြင်း။
2. **Cognee နှင့် MAF ပေါင်းစည်းခြင်း**: MAF အေဂျင့်တစ်ခုအား `@tool` functions အသုံးပြုပြီး Cognee ၏ သိပ္ပံဖြစ်ရပ်များကို မေးမြန်းခွင့်ပြုခြင်း။
3. **အစည်းအဝေးသတိထားစကားပြောဆိုခြင်း**: အစည်းအဝေးတစ်ခုတည်းအတွင်း မေးခွန်းများစွာဆက်လက်ဆက်သွယ်နိုင်ရန် စာသားအရင်းအမြစ်ကို ထိန်းသိမ်းထားခြင်း။
4. **ရှည်လျားသော အမှတ်တရ**: အရေးပါတဲ့ သိပ္ပံဖြစ်ရပ်များကို အစည်းအဝေးများကြားမှာ သိုလှောင်ထားပြီး ဆက်သွယ်မှုအသစ်များတွင် ပြန်လည်သုံးနိုင်ခြင်း။

### လိုအပ်ချက်များ
- Python 3.9+
- ဒေသတွင်း Redis လည်ပတ်နေမှု (`docker run -d -p 6379:6379 redis`) အစည်းအဝေးစီမံခန့်ခွဲမှုအတွက်
- LLM API key (ဥပမာ OpenAI) — `.env` တွင် `LLM_API_KEY` ကို သတ်မှတ်ရန်
- `.env` တွင် `CACHING=true` (Cognee sessions အတွက် လိုအပ်သည်)
- Azure AI Foundry project တစ်ခုနှင့် chat model တပ်ဆင်ပြီး ရှိနေခြင်း
- `.env` တွင် `AZURE_AI_PROJECT_ENDPOINT` နှင့် `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ဖြင့် authentication ပြီးသား (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agent မှတ်ဉာဏ် အမျိုးအစားများ

ဤ notebook သည် အဓိက Lesson 13 notebook တွင် ပါဝင်သော သုံးမျိုးသော မှတ်ဉာဏ် အမျိုးအစားများကိုပါ တူညီစွာ လေ့လာပြီး၊ Cognee ကို ရေရှည် မှတ်ဉာဏ် backend အဖြစ် အသုံးပြုသည်။

| မှတ်ဉာဏ် အမျိုးအစား | အလုပ်လုပ်ပုံ | အသက်တမ်း |
|---|---|---|
| **အလုပ်လုပ်နေသော** | `agent.create_session()` (MAF) | တစ်ချက် စကားစမြည် |
| **အတိုချုပ်** | Cognee session cache (Redis) | တစ်ချက် session |
| **ရေရှည်** | Cognee knowledge graph + vectors | အမြဲတမ်း |

### Cognee ၏ မှတ်ဉာဏ် တည်ဆောက်ပုံ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee စတိုးရန် ပြင်ဆင်ပါ


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — Building the Knowledge Base

ကျွန်ုပ်တို့၏ ကုဒ်ရေးသူ အကူအညီမည့် စနစ်အတွက် ဖလှယ်သိမြင်မှုအခြေခံများကို ဖန်တီးရန် ဒေတာအမျိုးအစားသုံးမျိုးကို စုဆောင်းပါသည်။

1. **Developer Profile** — ကိုယ်ပိုင်ကျွမ်းကျင်မှုနှင့် နည်းပညာနောက်ခံ
2. **Python Best Practices** — Python ၏ ဇန်နစ်နှင့် လက်တွေ့ လမ်းညွှန်များ
3. **Historical Conversations** — သမိုင်းထဲက ကိုယ်တိုင်နှင့် AI အကူအညီတွေကြား အမေးအဖြေတွေ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## နည်းပညာ ကုမ္ပဏီ ဗဟုသုတ ဇယားကို မြင်မြင်ကြင်ကြင် ဖော်ပြခြင်း

Cognee သည် ၎င်းထုတ်ယူထားသော အရာဝတ္ထုများနှင့် ဆက်နွယ်မှုများကို အပြန်အလှန် လှုပ်ရှားနိုင်သော HTML ဗျူအဂ်ဖြင့် ဖော်ပြနိုင်သည်။


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify ဖြင့် မှတ်ဉာဏ်ကို သန့်ရှင်းစွာ ဖြည့်တင်းခြင်း

`memify()` သည် ဗဟုသုတကြီးရတားကို ခွဲခြမ်းစိစစ်ပြီး အသိပညာရှိသော စည်းမျဉ်းများ ထုတ်လုပ်သည် — ပုံစံများ၊ အကောင်းဆုံး လေ့ကျင့်မှုများနှင့် အတွေးအခေါ်များအကြား ဆက်စပ်မှုများကို ထောက်လှမ်းလေ့လာသည်။


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## အပိုင်း 2 — Cognee ကိရိယာများနှင့် MAF Agent

အခုတော့ `@tool` function များမှတဆင့် Cognee ၏ အသိပညာကြားကွန်ရက်ကို မေးမြန်းနိုင်သော MAF agent တစ်ခုကို ဖန်တီးမည်။ ၎င်းက agent ကို ဆက်သွယ်ပြောဆိုမှု context များကို session များဖြင့် ထိန်းသိမ်းကာ graph အသိပညာဗေဒရှာဖွေရေး၏ အပြည့်အစုံစွမ်းအားကို အသုံးချခွင့်ပေးမည်ဖြစ်သည်။


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Sessions တို့နှင့်အတူ လုပ်ဆောင်မှုမှတ်ဉာဏ်

`AgentSession` (`agent.create_session()` ကနေဖန်တီးထားသော) သည် session အတွင်း လုပ်ဆောင်မှုမှတ်ဉာဏ်ကို ပံ့ပိုးပေးသည်။ agent သည် မျှော်မှန်းထားသည့် ယခင်ပို့စ်များကို ပြန်ရှာဖွေနိုင်ပြီး Cognee ၏ ရေရှည် အသိပညာ ဂရပ်ကိုလည်း မေးမြန်းနိုင်သည်။


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## New Session — Long-Term Memory Persists

အသစ်သောအစုံတစ်ခုစတင်ခြင်းသည် အလုပ်မှတ်ဉာဏ်ကို ဖယ်ရှားပစ်သွားသော်လည်း Cognee အချင်းချင်း နက်နဲသော ဗဟုသုတဂရပ်များကိုတော့ မူလတန်ဆိပ်သေးဆဲဖြစ်သည်။ အေးဂျင့်သည် ပြီးပြည့်စုံသော စကားပြောဆိုမှုအသစ်တစ်ခုတွင် တူညီသော ရေရှည်ဗဟုသုတကို ရယူနိုင်သည်။


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## အနှစ်ချုပ်

ဤ notebook တွင် သင်သည် **MAF ၏ စဉ်ဆက်မပြတ်မှတ်ဉာဏ်** (`agent.create_session()`) နှင့် **Cognee ၏ ရေရှည်သိထားမှု ဂရပ်** ကိုပေါင်းစပ်ထားသော ကုတ်ရေးရာအကူအညီကို ဖန်တီးလိုက်ပါသည်။

### သင်ယူထားသည့်အရာများ
1. **အသိပညာဂရပ် တည်ဆောက်ခြင်း**: Cognee သည် အစီအစဉ်မရှိသော စာသားများကို စုပ်ယူကာ ဂရပ်နှင့် vector မှတ်ဉာဏ် တည်ဆောက်သည်။
2. **memify ဖြင့် ဂရပ် ကိုတိုးချဲ့ခြင်း**: သင်၏ ရှိပြီးသားဂရပ်ပေါ်တွင် အင်္ဂါရပ်များနှင့် ပို၍ ချိတ်ဆက်နေထိုင်မှုများကို ထည့်သွင်းသည်။
3. **MAF + Cognee ပေါင်းစပ်မှု**: `@tool` ဖင့််ရှင်းများက MAF agent ကို Cognee ၏ ဂရပ်ကို သဘာဝအတိုင်း မေးမြန်းခွင့်ပေးသည်။
4. **စဉ်ဆက်မပြတ်မှတ်ဉာဏ် + ရေရှည်မှတ်ဉာဏ်**: `AgentSession` (`agent.create_session()` မှတဆင့်) သည် အစည်းအဝေးအခြေအနေကို ပံ့ပိုးပြီး Cognee သည် အမြဲတမ်းသိထားမှု ပံ့ပိုးသည်။
5. **NodeSets ဖြင့် စစ်ထုတ်ရှာဖွေရေး**: အသိပညာဂရပ်၏ အထူးပိုင်းအစ (ဥပမာ နိယာမများသာ) ကို ရည်ညွှန်း ။ 

### အဓိကယူဆချက်များ
- **Cognee** သည် မဖွဲ့စည်းထားသောစာသားများကို ဖွဲ့စည်းထားသော၊ ချိတ်ဆက်မှုသိထားသော မှတ်ဉာဏ်သို့ ပြောင်းလဲပေးသည် — ပြေပြစ် vector stock ထက် ပိုပြီး အင်အားကြီးသည်။
- **`@tool` ဖင့််ရှင်းများ** က MAF agents နှင့် ပြင်ပ အသိပညာစနစ်များ အကြား တိကျကျ ချိတ်ဆက်ပေးသည်။
- **`AgentSession`** (`agent.create_session()` မှတစ်ဆင့်) သည် တစ်စကားပြောစဉ်စဉ်ကို ရေရှည်မှတ်ဉာဏ်နှင့် ကွဲပြားစွာ ထိန်းသိမ်းပေးသည်။
- တူညီသော အသိပညာဂရပ်သည် session များနှင့် agent များစွာကို တစ်ပြိုင်နက် အသုံးပြုခွင့်ပြုသည်။

### တကယ့်ကမ္ဘာ သုံးခွင့်များ
- **Developer copilots**: ကုတ်သုံးသပ်ခြင်း၊ ဖြစ်ရပ်အခြေအနေလေ့လာခြင်း၊ ပရိုဂရမ်စတိုင် ကူညီသူများ
- **ဖောက်သည်ဆိုင်ရာ copilots**: ထုတ်ကုန်စာတမ်းများ၊ မေးခွန်းများ၊ CRM မှတ်တမ်းများအပေါ် စာပို့နေသူဆိုင်ရာ အကူအညီများ
- **အတွင်းရေး အထူးကျွမ်း ကျောက်သားများ**: မူဝါဒ၊ ဥပဒေ၊ လုံခြုံရေး အကူအညီများ guideline များအပေါ် အမြင်သက္ကာဖြင့် မေးမြန်းသူများ
- **သတင်းအချက်အလက် အတန်းလိုက် စုပေါင်းမှု**: ဖွဲ့စည်းပြီး မဖွဲ့စည်းထားသောဒေတာများကို တစ်ခုတည်းသော မေးမြန်းနိုင်သည့် ဂရပ်သို့ ပေါင်းစည်းမှု

### နောက်တစ်ကြိမ် ပြုလုပ်ရမည့်အရာများ
- Cognee တွင် အချိန်ဆိုင်ရာ အသိပညာစွမ်းရည် စမ်းသပ်မှုများ ပြုလုပ်ခြင်း
- ကဏ္ဍအပေါ် မူတည်သည့် OWL ontology တည်ဆောက်ခြင်း။
- အသုံးပြုသူ တုံ့ပြန်ချက်များ ဖြင့် ရှာဖွေမှုများ တိုးတက်အောင် ပြုလုပ်ခြင်း။
- Cognee မှတ်ဉာဏ် မျက်နှာပြင်တစ်ခုသုံး Multi-agent စနစ်များ အတိုက်အခံ တိုးချဲ့ခြင်း။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ဝန်ခံချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်စနစ်ဖြစ်သော [Co-op Translator](https://github.com/Azure/co-op-translator) ကို အသုံးပြု၍ ဘာသာပြန်ထားခြင်း ဖြစ်ပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှုကို ကြိုးပမ်းထားသော်လည်း၊ အလိုအလျောက် ဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မမှန်ကန်မှုများ ပါဝင်နိုင်သည်ကို သတိပြုပါရန် တောင်းဆိုအပ်ပါသည်။ မူရင်းစာတမ်းကို မိမိဘာသာစကားဖြင့်သာ အတည်ပြုရမည့် အဓိကအရင်းအမြစ်အဖြစ် မှတ်ယူကြရပါမည်။ အရေးကြီးသော အချက်အလက်များအတွက် သဘာဝကျသော လူသား ဘာသာပြန်သူ၏ ဘာသာပြန်ချက် အသုံးပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုရာမှ ဖြစ်ပေါ်လာသော ပြချက်မမှန်ခြင်း သို့မဟုတ် နားလည်မှားခြင်းများအတွက် ကျွန်ုပ်တို့သည် တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
